# Paper-Grade Evaluation: Cases A, B, and C

This notebook evaluates the existing ChatbotLP flow:

`natural prose -> semantic plan -> ProblemState -> validation -> solve -> reasoning/explanation`

It does not redesign the architecture. If Gemini is not configured, the run uses an explicit offline deterministic fixture demonstration and records that metadata in the tables. Live LLM mode fails visibly instead of substituting fixture states.

In [ ]:
import os

# Set GEMINI_API_KEY in your shell, VS Code/Jupyter environment, or a local .env loader
# before running this notebook. This notebook intentionally does not store API keys.
os.environ.setdefault("LLM_PROVIDER", "gemini")
os.environ.setdefault("GEMINI_MODEL", "gemini-3-flash-preview")


Set `GEMINI_API_KEY` in your local shell or VS Code/Jupyter environment to run live LLM interpretation. Leave it unset to run the notebook as an offline deterministic fixture demonstration. The notebook does not store a Gemini API key.

In [4]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.paper_grade_evaluation import (
    EVALUATION_PRESETS,
    build_evaluation_config,
    plot_error_categories,
    plot_passed_checks_by_case,
    run_paper_grade_evaluation,
)

pd.set_option("display.max_colwidth", 120)

## Quota-Safe Controls

Use the default `quota_safe_demo` settings for a small live Gemini run: three interpretation calls and no reasoning battery. Switch `evaluation_preset` or the stage toggles when you want a broader run.

In [5]:
list(EVALUATION_PRESETS)

['interpretation_only',
 'case_a_full',
 'case_b_interpretation',
 'case_c_interpretation',
 'reasoning_one_case',
 'full_evaluation',
 'quota_safe_demo']

In [35]:
# Presets: interpretation_only, case_a_full, case_b_interpretation,
# case_c_interpretation, reasoning_one_case, full_evaluation, quota_safe_demo.
evaluation_preset = "reasoning_one_case"

# Leave as an empty list to use the preset's case selection.
selected_cases = [
    "negative_bid_case_b",
]

run_interpretation = True
run_solver = True
run_reasoning_battery = True

# Use a smaller subset when run_reasoning_battery=True.
# Available ids: primal_lp, dual_lp, theorem_1_proof, strong_duality,
# dual_variable_interpretation, node_product_price_interpretation,
# complementary_slackness.
reasoning_prompt_subset = ["node_product_price_interpretation"]

## Run Evaluation

In [36]:
USE_LLM = bool(os.environ.get("GEMINI_API_KEY"))
evaluation_mode = "live_llm" if USE_LLM else "deterministic_fixture"
print(f"evaluation_mode = {evaluation_mode!r}")

config = build_evaluation_config(
    preset=evaluation_preset,
    selected_cases=tuple(selected_cases) if selected_cases else None,
    run_interpretation=run_interpretation,
    attempt_solve=run_solver,
    run_reasoning=run_reasoning_battery,
    reasoning_prompt_subset=tuple(reasoning_prompt_subset) if reasoning_prompt_subset else None,
    use_llm=USE_LLM,
    use_llm_for_reasoning=USE_LLM,
    use_deterministic_fixture=not USE_LLM,
    mode="guided",
)

report = run_paper_grade_evaluation(config=config)
tables = report["tables"]

report["metadata"]

{'llm_provider': 'gemini',
 'gemini_model': 'gemini-3-flash-preview',
 'gemini_configured': True,
 'config': {'mode': 'guided',
  'selected_cases': ('negative_bid_case_b',),
  'run_interpretation': True,
  'use_llm': True,
  'use_llm_for_reasoning': True,
  'use_deterministic_fixture': False,
  'attempt_solve': True,
  'run_reasoning': True,
  'reasoning_prompt_subset': ('node_product_price_interpretation',)}}

## LLM and Fixture Metadata

In [37]:
metadata_table = pd.DataFrame(
    [
        {
            "case": case["name"],
            "interpretation_source": case["interpretation_metadata"].get("interpretation_source"),
            "deterministic_fixture_used": case["interpretation_metadata"].get("deterministic_fixture_used"),
            "failure_type": case["interpretation_metadata"].get("failure_type"),
            "llm_failure": case["interpretation_metadata"].get("llm_failure"),
            "llm_provider": case["interpretation_metadata"].get("llm_provider"),
            "gemini_model": case["interpretation_metadata"].get("gemini_model"),
        }
        for case in report["cases"]
    ]
)
display(metadata_table)

,case,interpretation_source,deterministic_fixture_used,failure_type,llm_failure,llm_provider,gemini_model
0,negative_bid_case_b,live_llm_pipeline,False,None,None,gemini,gemini-3-flash-preview


## Output Tables

In [38]:
for table_name in [
    "case_level_summary",
    "interpretation_accuracy",
    "solver_readiness_accuracy",
    "solve_accuracy",
    "reasoning_prompt_success",
    "error_category_counts",
]:
    display(Markdown(f"### {table_name.replace('_', ' ').title()}"))
    display(tables[table_name])

### Case Level Summary

,case,label,family,semantic_plan_created,problem_state_created,structural_match,blocking_error_count,benign_extra_name_fields,solver_ready_expected,solver_ready_actual,solver_ready_correct,solve_expected,solve_success,objective_match,passed_checks,total_applicable_checks
0,negative_bid_case_b,Negative-Bid Case B,Case B,True,True,True,0,3,True,True,True,True,True,True,9,9


### Interpretation Accuracy

,case,structural_match,missing_fields,wrong_numeric_values,wrong_ownership_relations,wrong_node_product_mappings,extra_invented_entities,benign_extra_name_fields,blocking_error_categories
0,negative_bid_case_b,True,0,0,0,0,0,3,


### Solver Readiness Accuracy

...

,case,solve_expected,solve_success,status,solver_name,expected_objective,actual_objective,objective_match,accepted_bid_match,transport_flow_match,technology_activity_match,solver_message
0,negative_bid_case_b,True,True,optimal,None,240.0,240.0,True,True,True,True,Solver glpk terminated with status optimal (termination condition: optimal)


### Reasoning Prompt Success

,case,prompt_id,prompt_label,success,intent,render_mode,response_source,fallback_triggered,fallback_reason,llm_exception_type,raw_llm_output_present,llm_output_length,validation_warnings,validation_fatal,response_preview
0,negative_bid_case_b,node_product_price_interpretation,Node-product price interpretation,False,explanation,markdown,deterministic,True,llm_exception,RuntimeError,False,0,[],"[Gemini API call failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota...",Paper-grounded explanation\n\nIntuition: node-product prices summarize local scarcity in the coordinated clearing sy...


### Error Category Counts

,category,count
0,benign_extra_name_field,3


## Figures

In [ ]:
generated_figures = {}

fig, ax = plt.subplots(figsize=(10, 4))
plot_passed_checks_by_case(tables["case_level_summary"], ax=ax)
fig.tight_layout()
generated_figures["passed_checks_by_case"] = fig
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
plot_error_categories(tables["error_category_counts"], ax=ax)
fig.tight_layout()
generated_figures["error_categories_across_cases"] = fig
plt.show()


## Inspect Blocking Errors

In [11]:
blocking_error_rows = []
for case in report["cases"]:
    for error in case["comparison"]["blocking_errors"]:
        blocking_error_rows.append({"case": case["name"], **error})

display(pd.DataFrame(blocking_error_rows))

""


## Paper Summary Export

Write paper-ready CSV tables, generated figures, compact run metadata, and a concise summary paragraph to `paper_grade_outputs/`.


In [ ]:
import json
from datetime import datetime, timezone

output_dir = repo_root / "paper_grade_outputs"
output_dir.mkdir(parents=True, exist_ok=True)

paper_tables = dict(tables)
paper_tables["interpretation_metadata"] = metadata_table
paper_tables["blocking_error_details"] = pd.DataFrame(blocking_error_rows)

csv_paths = {}
for table_name, table in paper_tables.items():
    csv_path = output_dir / f"{table_name}.csv"
    table.to_csv(csv_path, index=False)
    csv_paths[table_name] = str(csv_path.relative_to(repo_root))

figure_paths = {}
for figure_name, figure in globals().get("generated_figures", {}).items():
    if figure is None:
        continue
    png_path = output_dir / f"{figure_name}.png"
    figure.savefig(png_path, dpi=160, bbox_inches="tight")
    figure_paths[figure_name] = str(png_path.relative_to(repo_root))

case_summary = tables["case_level_summary"]
solve_accuracy = tables["solve_accuracy"]
reasoning_success = tables.get("reasoning_prompt_success", pd.DataFrame())

live_interpretation_cases = int(
    (
        metadata_table["interpretation_source"].eq("live_llm_pipeline")
        & metadata_table["deterministic_fixture_used"].ne(True)
    ).sum()
)
deterministic_fixture_cases = int(metadata_table["deterministic_fixture_used"].eq(True).sum())
structural_match_cases = int(case_summary["structural_match"].eq(True).sum())
zero_blocking_error_cases = int(case_summary["blocking_error_count"].eq(0).sum())
solver_ready_correct_cases = int(case_summary["solver_ready_correct"].eq(True).sum())
objective_match_cases = int(solve_accuracy["objective_match"].eq(True).sum())

if reasoning_success.empty:
    live_reasoning_prompts = 0
    successful_reasoning_prompts = 0
    llm_reasoning_without_deterministic_response_prompts = 0
else:
    live_reasoning_prompts = int(reasoning_success["response_source"].eq("llm").sum())
    successful_reasoning_prompts = int(reasoning_success["success"].eq(True).sum())
    llm_reasoning_without_deterministic_response_prompts = int(reasoning_success["fallback_triggered"].eq(False).sum())

provider = report["metadata"].get("llm_provider") or os.getenv("LLM_PROVIDER") or "unset"
model = report["metadata"].get("gemini_model") or os.getenv("GEMINI_MODEL") or "unset"

source_note = (
    "live LLM calls"
    if live_interpretation_cases or live_reasoning_prompts
    else "deterministic_fixture_mode"
)
if deterministic_fixture_cases and (live_interpretation_cases or live_reasoning_prompts):
    source_note = "a mix of live LLM calls and deterministic_fixture_mode"

run_summary = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "provider": provider,
    "model": model,
    "gemini_configured": bool(report["metadata"].get("gemini_configured")),
    "config": report["metadata"].get("config", {}),
    "result_source_note": source_note,
    "counts": {
        "cases_evaluated": int(len(case_summary)),
        "live_llm_interpretation_cases": live_interpretation_cases,
        "deterministic_fixture_cases": deterministic_fixture_cases,
        "structural_match_true_cases": structural_match_cases,
        "blocking_error_count_zero_cases": zero_blocking_error_cases,
        "solver_ready_correct_true_cases": solver_ready_correct_cases,
        "solve_objective_match_true_cases": objective_match_cases,
        "reasoning_prompts": int(len(reasoning_success)),
        "live_llm_reasoning_prompts": live_reasoning_prompts,
        "reasoning_success_true_prompts": successful_reasoning_prompts,
        "reasoning_without_deterministic_response_prompts": llm_reasoning_without_deterministic_response_prompts,
    },
    "interpretation_source_counts": metadata_table["interpretation_source"].value_counts(dropna=False).to_dict(),
    "reasoning_response_source_counts": (
        reasoning_success["response_source"].value_counts(dropna=False).to_dict()
        if not reasoning_success.empty
        else {}
    ),
    "csv_outputs": csv_paths,
    "figure_outputs": figure_paths,
}

summary_path = output_dir / "run_summary.json"
summary_path.write_text(json.dumps(run_summary, indent=2, default=str), encoding="utf-8")

summary_paragraph = (
    f"Paper-grade ABC evaluation used provider {provider} with model {model}. "
    f"The current run includes {live_interpretation_cases} live LLM interpretation case(s) "
    f"and {deterministic_fixture_cases} deterministic_fixture_mode case(s), so the reported results come from {source_note}. "
    f"Across {len(case_summary)} evaluated case(s), structural_match=True for {structural_match_cases}, "
    f"blocking_error_count=0 for {zero_blocking_error_cases}, solver_ready_correct=True for "
    f"{solver_ready_correct_cases}, and solve objective_match=True for {objective_match_cases}. "
    f"The reasoning battery includes {len(reasoning_success)} prompt(s), including "
    f"{live_reasoning_prompts} live LLM reasoning prompt(s); success=True for "
    f"{successful_reasoning_prompts}, and no deterministic response substitution for {llm_reasoning_without_deterministic_response_prompts}."
)

print(summary_paragraph)
print(f"Saved {len(csv_paths)} CSV table(s), {len(figure_paths)} PNG figure(s), and run metadata to {output_dir.relative_to(repo_root)}/")
